# Exercise 2.10: Sampling and Reconstruction

Beginning with a time continuous signal $x_{\mathrm{K}}(t)$ and the
properties of the fourier transform (2.4.1, 2.4.2), we get a
series $x(k)$ by sampling with frequency $f_A$ and the
transformation pair (2.3.11, 2.3.12). The spectra
$X_{\mathrm{K}}(j\omega)$ and $X(e^{j\Omega})$ are linked by 
expression (2.4.8):

$$
X(e^{j\Omega})=\frac{1}{T_A}\sum_{i=-\infty}^{\infty} X_{\mathrm{K}}\left(j\frac{\Omega+i2\pi}{T_A}\right)
$$

Thus, the spectrum of a discrete signal $X(e^{j\Omega})$ can be
obtained by overlapping spectra of a continuous time signal
$X_{\mathrm{K}}(j\omega)$, which are shifted by multiples of $2\pi$ and
normalized with $1/T_A$.  

To reconstruct the continuous time out of a discrete time signal,
the sub spectra $X_{\mathrm{K}}(j(\Omega+i2\pi)/T)$ must not
overlap[^1]. This results in the demands of the so called **sampling theorem** by Shannon:

[^1]: The case where these spectra overlap is called *aliasing*.

* $X_{\mathrm{K}}(j\omega)$ has to be band limited — thus vanish in the interval
  $|\omega| \ge \omega_{\text{max}}$.
* The sampling frequency $\omega_A = 2\pi f_A = 2\pi /T_A$ has to be twice
  the value of the maximum frequency $\omega_{\text{max}}$ of
  $X_{\mathrm{K}}(j\omega)$.

---

To study sampling effects, it is necessary to simulate the time continuous
signal with a time discrete series. Thus studying these effects, two sampling
periods are in use.

One sampling period $\Delta t$ is used to simulate the
continuous time signal. The other sampling period $T_A$ is used to
sample the "analogous" signal.  
The effects of sampling can be easily seen in frequency domain.
To simulate the Fourier transform of a
continuous time signal, function `lafplot()`  
(see `helper_functions.py` for further help) is used.

## a) Simulate the following signal

$$
s(t) = \cos(2\pi f_0 t + \phi), \quad 0 \leq t \leq T
$$

The signal frequency shall be $f_0 = 2\ \mathrm{kHz}$ and the sampling frequency  
$\frac{1}{\Delta t} = 80\ \mathrm{kHz}$.  
Choose $T$ so that the signal contains about 2000 samples of the simulated analogous signal.  

Plot the "analogous" signal using the `plt.plot()` function. Pay attention to correct scaling of the time axis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import freqz, cheby2, filtfilt
from helper_functions import lzerofill, lafplot, ldtft

# BEGIN SOLUTION
fsim = 80000                            # Sampling Frequency
delta_t = np.arange(0, 0.02, 1/fsim)    # Time Vector
f0 = 2000                               # Frequency of the Cosine Wave
s = np.cos(2 * np.pi * f0 * delta_t)    # Cosine signal

# Plotting
plt.figure()
plt.plot(delta_t, s)
plt.xlabel('Time (sec)')
plt.ylabel('Amplitude')
plt.title('Analog Signal')
plt.show()
# END SOLUTION

## b) Plot the spectrum of the "analogous" signal

Use the function `lafplot()` to plot the spectrum.

In [ ]:
# BEGIN SOLUTION
lafplot(s, 1/fsim)
# END SOLUTION

## c) Sampling with A/D–converter

An A/D–converter samples an analogous signal with period $T_A$.  
Simulate this behavior by sampling the analogous signal with a frequency of $f_A = 8\ \mathrm{kHz}$ and plotting the result.

In [ ]:
# BEGIN SOLUTION
fa = 8000                           # Sampling Frequency
l = fsim // fa                      # Downsampling Factor
index = np.arange(1, len(s), l)     # Indexing
sd = s[index]                       # (Re)sampled Signal

# Plotting
plt.figure()
plt.stem(sd[:50])
plt.grid()
plt.xlabel('k')
plt.ylabel('Amplitude')
plt.title('Discrete Signal')
plt.show()
# END SOLUTION

## d) Calculate the spectrum of the sampled signal

Use the function `ldtft()` (see `helper_functions.py`).

In [ ]:
# BEGIN SOLUTION
H, W = ldtft(sd, 512)

# Plotting
plt.figure()
plt.plot(W / np.pi, np.abs(H))
plt.grid()
plt.xlabel('Omega/pi')
plt.ylabel('Amplitude')
plt.title('Discrete Fourier Transform (Magnitude)')
plt.show()
# END SOLUTION

## e) Design a low-pass filter to reconstruct the analogous signal

This analogous reconstruction low-pass filter has to be simulated again.  
Use the Python equivalent function (e.g. from `scipy.signal.cheby2`): 

```python
b, a = cheby2(9, 60, fcut)
```

This designs a low-pass of 9th order with $60\ \mathrm{dB}$ stop-band attenuation.  
The analogous cut-off frequency `fcut` shall be $\frac{1}{2}f_A$.  
In MATLAB, normalize this frequency:  

```
fcut = fsamp / fsim
```

Plot the transfer function of the low-pass using the function `freqz()`.

In [ ]:
# BEGIN SOLUTION
fcut = fa / fsim
b, a = cheby2(9, 60, fcut)
W, H = freqz(b, a, 128, fsim)

# Plotting
# Frequency Response
plt.figure()
plt.semilogy(W, np.abs(H))
plt.grid()
plt.xlabel('Frequency in Hz')
plt.ylabel('Magnitude')
plt.title('Reconstruction Chebyshev TP')
plt.show()

# Phase Response
plt.figure()
plt.plot(W, np.angle(H))
plt.grid()
plt.xlabel('Frequency in Hz')
plt.ylabel('Phase')
plt.title('Reconstruction Chebyshev TP')
plt.show()
# END SOLUTION

## f) D/A–Converter reconstruction

A D/A–Converter generates a staircase analogous time domain signal.  
This signal is smoothed by the designed Chebyshev low-pass filter.  
To simulate this process, zeros are added between the samples and the signal is filtered using the designed reconstruction low-pass filter.  

Plot the reconstructed analogous signal in time and frequency domain.  

**Python hint:** A function like `lzerofill()` (which adds the desired number of zeros between elements of a vector) can be implemented or imported from `helper_functions.py`.

In [ ]:
# BEGIN SOLUTION
sa = lzerofill(sd, l)  # Reconstruction/Inserting Zeros

sa = sa.flatten()
sr = filtfilt(b, a, sa)  # Reconstruction

# Plotting
# Reconstructed Sine Wave
plt.figure()
plt.plot(sr)
plt.title('Reconstructed Sine Wave')
plt.xlabel('Time in Samples')
plt.ylabel('Amplitude')
plt.show()

# Amplitude
lafplot(sr, 1/fsim)
plt.ylabel('Amplitude')
plt.show()
# END SOLUTION